# Качество обращений NYC 311

ЛР № 3, переход из raw в staging

Основной вопрос Project Brief касается числа обращений и времени до закрытия по типам проблем и агентствам в Манхэттене за январь 2025 года.
Здесь проверяется пригодность данных для этих расчётов, бизнес-метрики не рассчитываются

Одна строка представляет обращение, бизнес-ключ `unique_key`.
Используется снимок ЛР № 2 из восьми полей, профиль строится для пяти полей, необходимых основному вопросу

In [1]:
from pathlib import Path

RAW_PATH = "s3://raw/nyc311_manhattan/ingested_on=2026-10-14/nyc311_manhattan_2025-01.csv"
PERIOD_START = "2025-01-01"
PERIOD_END = "2025-02-01"
OUTPUT_PREFIX = "s3://staging/nyc311_manhattan/source_ingested_on=2026-10-14"
STAGING_PATH = f"{OUTPUT_PREFIX}/staging.parquet"
QUARANTINE_PATH = f"{OUTPUT_PREFIX}/quarantine.parquet"
PROFILE_FIELDS = ["unique_key", "created_date", "closed_date", "complaint_type", "agency"]

assert RAW_PATH.split("ingested_on=")[1].split("/")[0] == OUTPUT_PREFIX.rsplit("=", 1)[1]
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

## Подключение и исходные данные

Настройки S3 загружаются из локального `.env`, ключи доступа не выводятся.
DuckDB читает CSV непосредственно из S3, локальная резервная копия не используется.
Исходные поля читаются как текст, чтобы сохранить их значения в карантине.
Предполагаемые типы определяются отдельно автоматическим чтением DuckDB

In [2]:
import hashlib
import os
from html import escape
from urllib.parse import urlsplit

import boto3
import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

load_dotenv(PROJECT_ROOT / ".env")
con = duckdb.connect(":memory:")
con.execute("INSTALL httpfs FROM 'https://extensions.duckdb.org'")
con.execute("LOAD httpfs")
settings = {
    "s3_endpoint": os.environ["S3_ENDPOINT"],
    "s3_access_key_id": os.environ["S3_ACCESS_KEY"],
    "s3_secret_access_key": os.environ["S3_SECRET_KEY"],
    "s3_region": "us-east-1",
    "s3_url_style": "path",
    "s3_use_ssl": False,
}
for name, value in settings.items():
    con.execute(f"SET {name} = ?", [value])
s3 = boto3.client(
    "s3",
    endpoint_url=f"http://{os.environ['S3_ENDPOINT']}",
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    region_name="us-east-1",
)

In [3]:
def show_rows(columns, rows):
    header = "".join(f"<th>{escape(str(value))}</th>" for value in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))


def show_query(sql, params=None):
    result = con.execute(sql, params or [])
    rows = result.fetchall()
    show_rows([column[0] for column in result.description], rows)
    return rows


def raw_fingerprint():
    parsed = urlsplit(RAW_PATH)
    response = s3.get_object(Bucket=parsed.netloc, Key=parsed.path.lstrip("/"))
    with response["Body"] as body:
        digest = hashlib.sha256(body.read()).hexdigest()
    return {
        "sha256": digest,
        "bytes": response["ContentLength"],
        "etag": response["ETag"],
        "last_modified": response["LastModified"].isoformat(),
    }


raw_before = raw_fingerprint()
inferred = con.read_csv(RAW_PATH, header=True, hive_partitioning=False)
inferred_types = dict(zip(inferred.columns, map(str, inferred.types), strict=True))
con.read_csv(RAW_PATH, header=True, all_varchar=True, hive_partitioning=False).create_view(
    "raw_input"
)
show_query("SELECT * FROM raw_input ORDER BY unique_key LIMIT 3")
print("SHA256", raw_before["sha256"])

unique_key,created_date,closed_date,complaint_type,agency,borough,status,open_data_channel_type
63572326,2025-01-01T00:38:37.000,2025-01-01T01:14:24.000,Noise - Commercial,NYPD,MANHATTAN,Closed,ONLINE
63572331,2025-01-01T00:37:52.000,2025-01-01T00:50:50.000,Illegal Fireworks,NYPD,MANHATTAN,Closed,ONLINE
63572341,2025-01-01T00:59:00.000,2025-01-01T01:13:08.000,Noise - Commercial,NYPD,MANHATTAN,Closed,ONLINE


SHA256 645bfe253cf4705f35d9381cc26d456b722c18e0bbc96562b4d7e54f247294d8


## Профиль

Профиль рассчитан до нормализации и исключения строк.
Пропуском считается NULL или пустой текст после удаления краевых пробелов.
Для каждого выбранного поля знаменателем доли пропусков служат все строки raw.
Тип в таблице определён автоматическим чтением DuckDB

In [4]:
raw_count = con.execute("SELECT count(*) FROM raw_input").fetchone()[0]
raw_field_count = len(con.execute("DESCRIBE raw_input").fetchall())
profile_rows = []
for field in PROFILE_FIELDS:
    checked, missing, examples = con.execute(f"""
        SELECT count(*),
            count(*) FILTER (WHERE nullif(trim({field}), '') IS NULL),
            array_slice(list(DISTINCT {field} ORDER BY {field})
                FILTER (WHERE {field} IS NOT NULL), 1, 2)
        FROM raw_input
    """).fetchone()
    profile_rows.append(
        [
            field,
            inferred_types[field],
            "; ".join(examples),
            checked,
            missing,
            round(100 * missing / checked, 4),
        ]
    )
print(f"Строк {raw_count}, полей {raw_field_count}")
show_rows(["Поле", "Тип DuckDB", "Примеры", "Проверено", "Пропуски", "Пропуски, %"], profile_rows)

Строк 61331, полей 8


Поле,Тип DuckDB,Примеры,Проверено,Пропуски,"Пропуски, %"
unique_key,BIGINT,63572326; 63572331,61331,0,0.0
created_date,TIMESTAMP,2025-01-01T00:00:37.000; 2025-01-01T00:01:06.000,61331,0,0.0
closed_date,TIMESTAMP,2025-01-01T00:07:59.000; 2025-01-01T00:15:27.000,61331,1036,1.6892
complaint_type,VARCHAR,AHV Inspection Unit; APPLIANCE,61331,0,0.0
agency,VARCHAR,DCWP; DEP,61331,0,0.0


Повторы считаются как сумма `число строк ключа − 1` по непустым ключам после удаления краевых пробелов.
Это число избыточных строк, а не число всех участников повторяющихся групп.
Допустимость ключа и дат проверяется ниже отдельными правилами

In [5]:
show_query("""
    WITH keys AS (
        SELECT nullif(trim(unique_key), '') AS request_key, count(*) AS n
        FROM raw_input
        WHERE nullif(trim(unique_key), '') IS NOT NULL
        GROUP BY request_key
    )
    SELECT count(*) AS distinct_nonempty_keys,
        coalesce(sum(greatest(n - 1, 0)), 0) AS duplicate_excess_rows
    FROM keys
""")
show_query("""
    SELECT min(try_cast(created_date AS TIMESTAMP)) AS first_created_at,
           max(try_cast(created_date AS TIMESTAMP)) AS last_created_at
    FROM raw_input
""")

distinct_nonempty_keys,duplicate_excess_rows
61331,0


first_created_at,last_created_at
2025-01-01 00:00:37,2025-01-31 23:59:48


[(datetime.datetime(2025, 1, 1, 0, 0, 37),
  datetime.datetime(2025, 1, 31, 23, 59, 48))]

Распределение по агентствам показывает, на какие группы придётся основная часть будущего сравнения.
Показаны пять наиболее частых значений, остальные объединены только в таблице профиля.
Исходные записи и выходные файлы не агрегируются

In [6]:
agency_distribution = show_query("""
    WITH counts AS (
        SELECT agency, count(*) AS n FROM raw_input GROUP BY agency
    ), ranked AS (
        SELECT *, row_number() OVER (ORDER BY n DESC, agency) AS position FROM counts
    ), groups AS (
        SELECT CASE WHEN position <= 5 THEN agency ELSE 'Остальные' END AS agency_group,
            n, CASE WHEN position <= 5 THEN position ELSE 6 END AS position
        FROM ranked
    )
    SELECT agency_group, sum(n)::BIGINT AS rows,
        round(100.0 * sum(n) / (SELECT count(*) FROM raw_input), 4) AS percent
    FROM groups GROUP BY agency_group, position ORDER BY position
""")

agency_group,rows,percent
HPD,24333,39.6749
NYPD,17656,28.7881
DSNY,4642,7.5688
DEP,3239,5.2812
DOT,2839,4.629
Остальные,8622,14.0581


## Правила качества

Все пороги заданы числом нарушений, допустимое значение равно нулю.
Сравнение выполняется в коде, превышение означает `нарушения > порог`.
Действие применяется к каждой проблемной строке независимо от общего результата проверки

Пустая дата закрытия сама по себе допустима и не восстанавливается по статусу.
Она нужна для анализа доли обращений без даты закрытия из Project Brief.
Для подготовки проверок даты однозначно приводятся к TIMESTAMP, часовой пояс не придумывается

In [7]:
con.execute("""
    CREATE TEMP TABLE normalized AS
    SELECT *,
        nullif(trim(unique_key), '') AS request_key,
        try_cast(nullif(trim(created_date), '') AS TIMESTAMP) AS created_at,
        try_cast(nullif(trim(closed_date), '') AS TIMESTAMP) AS closed_at,
        nullif(trim(complaint_type), '') AS complaint_type_clean,
        nullif(trim(agency), '') AS agency_clean,
        count(*) OVER (PARTITION BY nullif(trim(unique_key), '')) AS key_count
    FROM raw_input
""")
quality_results = []


def record_result(rule_id, counts, threshold, severity, action):
    checked, violations = map(int, counts)
    assert isinstance(threshold, int) and threshold >= 0
    result = {
        "id": rule_id,
        "checked": checked,
        "violations": violations,
        "violation_share": violations / checked if checked else None,
        "threshold_rows": threshold,
        "threshold_exceeded": violations > threshold,
        "severity": severity,
        "action": action,
    }
    quality_results.append(result)
    show_rows(list(result), [list(result.values())])

### 1. `unique_request_key`

Поле `unique_key`, проверяются все строки.
Ключ должен быть заполнен, состоять из цифр и встречаться ровно один раз.
Порог 0 строк, критичность `critical`, действие `quarantine`

Повторы завышают число обращений, пустой ключ не позволяет проверить гранулярность.
В карантин попадают все строки повторяющегося ключа, произвольный выбор одной копии не выполняется

In [8]:
threshold_unique_request_key = 0
counts = con.execute("""
    SELECT count(*), count(*) FILTER (
        WHERE request_key IS NULL OR NOT regexp_full_match(request_key, '[0-9]+')
            OR key_count > 1
    )
    FROM normalized
""").fetchone()
record_result("unique_request_key", counts, threshold_unique_request_key, "critical", "quarantine")

id,checked,violations,violation_share,threshold_rows,threshold_exceeded,severity,action
unique_request_key,61331,0,0.0,0,False,critical,quarantine


### 2. `created_in_period`

Поле `created_date`, проверяются все строки.
Дата должна быть заполнена, распознаваться как TIMESTAMP и входить в полуинтервал `[2025-01-01, 2025-02-01)`.
Порог 0 строк, критичность `critical`, действие `quarantine`

Дата определяет период анализа, день недели и час регистрации из Project Brief.
Строка с неизвестной датой или за пределами периода не подходит для этих сравнений

In [9]:
threshold_created_in_period = 0
counts = con.execute(
    """
    SELECT count(*), count(*) FILTER (
        WHERE created_at IS NULL OR created_at < CAST(? AS TIMESTAMP)
            OR created_at >= CAST(? AS TIMESTAMP)
    )
    FROM normalized
""",
    [PERIOD_START, PERIOD_END],
).fetchone()
record_result("created_in_period", counts, threshold_created_in_period, "critical", "quarantine")

id,checked,violations,violation_share,threshold_rows,threshold_exceeded,severity,action
created_in_period,61331,0,0.0,0,False,critical,quarantine


### 3. `complaint_type_present`

Поле `complaint_type`, проверяются все строки.
Тип проблемы должен быть непустым после удаления краевых пробелов.
Порог 0 строк, критичность `warning`, действие `flag`

Пропуск мешает сравнению типов проблем, но не делает само обращение недействительным.
Такая строка остаётся в staging с `quality_missing_complaint_type = true`, значение не угадывается

In [10]:
threshold_complaint_type_present = 0
counts = con.execute("""
    SELECT count(*), count(*) FILTER (WHERE complaint_type_clean IS NULL)
    FROM normalized
""").fetchone()
record_result("complaint_type_present", counts, threshold_complaint_type_present, "warning", "flag")

id,checked,violations,violation_share,threshold_rows,threshold_exceeded,severity,action
complaint_type_present,61331,0,0.0,0,False,warning,flag


### 4. `agency_present`

Поле `agency`, проверяются все строки.
Агентство должно быть непустым после удаления краевых пробелов.
Порог 0 строк, критичность `warning`, действие `flag`

Без агентства нельзя отнести обращение к группе для сравнения объёмов и сроков.
Строка сохраняется с `quality_missing_agency = true`, неизвестное агентство не подставляется

In [11]:
threshold_agency_present = 0
counts = con.execute("""
    SELECT count(*), count(*) FILTER (WHERE agency_clean IS NULL)
    FROM normalized
""").fetchone()
record_result("agency_present", counts, threshold_agency_present, "warning", "flag")

id,checked,violations,violation_share,threshold_rows,threshold_exceeded,severity,action
agency_present,61331,0,0.0,0,False,warning,flag


### 5. `valid_closure_order`

Связь `created_date` и `closed_date`.
Проверяются строки с непустой исходной датой закрытия и распознанной датой создания.
Дата закрытия должна распознаваться и быть не раньше создания.
Порог 0 строк, критичность `critical`, действие `quarantine`

Отрицательные интервалы искажают время до закрытия из Project Brief.
Противоречивое обращение целиком исключается из staging, поэтому его будущие подсчёты относятся к допустимым записям, а не ко всему raw.
Строки без даты закрытия этим правилом не исключаются

In [12]:
threshold_valid_closure_order = 0
counts = con.execute("""
    SELECT count(*), count(*) FILTER (
        WHERE closed_at IS NULL OR closed_at < created_at
    )
    FROM normalized
    WHERE nullif(trim(closed_date), '') IS NOT NULL AND created_at IS NOT NULL
""").fetchone()
record_result(
    "valid_closure_order", counts, threshold_valid_closure_order, "critical", "quarantine"
)

id,checked,violations,violation_share,threshold_rows,threshold_exceeded,severity,action
valid_closure_order,60295,27,0.0004477983249025624,0,True,critical,quarantine


### Сводка правил

Доля нарушений рассчитана от числа строк, к которым применяется конкретное правило.
Для `valid_closure_order` строки с отсутствующей датой закрытия не входят в знаменатель

In [13]:
show_rows(list(quality_results[0]), [list(result.values()) for result in quality_results])
show_query("""
    SELECT unique_key, created_date, closed_date, complaint_type, agency
    FROM normalized
    WHERE nullif(trim(closed_date), '') IS NOT NULL AND created_at IS NOT NULL
        AND (closed_at IS NULL OR closed_at < created_at)
    ORDER BY unique_key LIMIT 2
""")

id,checked,violations,violation_share,threshold_rows,threshold_exceeded,severity,action
unique_request_key,61331,0,0.0,0,False,critical,quarantine
created_in_period,61331,0,0.0,0,False,critical,quarantine
complaint_type_present,61331,0,0.0,0,False,warning,flag
agency_present,61331,0,0.0,0,False,warning,flag
valid_closure_order,60295,27,0.0004477983249025624,0,True,critical,quarantine


unique_key,created_date,closed_date,complaint_type,agency
63656893,2025-01-06T07:51:00.000,2025-01-04T02:00:00.000,Street Light Condition,DOT
63748892,2025-01-13T09:22:00.000,2025-01-10T23:50:00.000,Street Light Condition,DOT


[('63656893',
  '2025-01-06T07:51:00.000',
  '2025-01-04T02:00:00.000',
  'Street Light Condition',
  'DOT'),
 ('63748892',
  '2025-01-13T09:22:00.000',
  '2025-01-10T23:50:00.000',
  'Street Light Condition',
  'DOT')]

## Подготовка staging и карантина

В staging идентификатор остаётся текстом, даты становятся TIMESTAMP.
В текстовых полях удаляются краевые пробелы, пустой текст становится NULL.
Имена восьми исходных полей сохраняются, добавляются два логических признака пропусков.
Неизвестные даты, категории и часовой пояс не восстанавливаются

Для карантина сохраняются все восемь исходных полей без нормализации.
`quality_rule_ids` представляет список строк VARCHAR[], по одному id на каждое критическое нарушение.
`quarantine_reason` объединяет все причины через `; `.
Каждая исходная строка проходит один отбор, карантин имеет приоритет перед флагами

In [14]:
con.execute(
    """
    CREATE TEMP TABLE checked AS
    SELECT *,
        complaint_type_clean IS NULL AS quality_missing_complaint_type,
        agency_clean IS NULL AS quality_missing_agency,
        list_filter([
            CASE WHEN request_key IS NULL OR NOT regexp_full_match(request_key, '[0-9]+')
                OR key_count > 1 THEN 'unique_request_key' END,
            CASE WHEN created_at IS NULL OR created_at < CAST(? AS TIMESTAMP)
                OR created_at >= CAST(? AS TIMESTAMP) THEN 'created_in_period' END,
            CASE WHEN nullif(trim(closed_date), '') IS NOT NULL AND created_at IS NOT NULL
                AND (closed_at IS NULL OR closed_at < created_at) THEN 'valid_closure_order' END
        ], rule_id -> rule_id IS NOT NULL) AS quality_rule_ids,
        concat_ws('; ',
            CASE WHEN request_key IS NULL OR NOT regexp_full_match(request_key, '[0-9]+')
                OR key_count > 1 THEN 'Пустой, нечисловой или повторяющийся ключ обращения' END,
            CASE WHEN created_at IS NULL OR created_at < CAST(? AS TIMESTAMP)
                OR created_at >= CAST(? AS TIMESTAMP)
                THEN 'Дата создания отсутствует, не распознана или вне периода' END,
            CASE WHEN nullif(trim(closed_date), '') IS NOT NULL AND created_at IS NOT NULL
                AND (closed_at IS NULL OR closed_at < created_at)
                THEN 'Дата закрытия не распознана или раньше создания' END
        ) AS quarantine_reason
    FROM normalized
""",
    [PERIOD_START, PERIOD_END, PERIOD_START, PERIOD_END],
)
con.execute("""
    CREATE TEMP VIEW staging_output AS
    SELECT request_key AS unique_key, created_at AS created_date, closed_at AS closed_date,
        complaint_type_clean AS complaint_type, agency_clean AS agency,
        nullif(trim(borough), '') AS borough,
        nullif(trim(status), '') AS status,
        nullif(trim(open_data_channel_type), '') AS open_data_channel_type,
        quality_missing_complaint_type, quality_missing_agency
    FROM checked WHERE len(quality_rule_ids) = 0
""")
con.execute("""
    CREATE TEMP VIEW quarantine_output AS
    SELECT unique_key, created_date, closed_date, complaint_type, agency, borough,
        status, open_data_channel_type, quality_rule_ids, quarantine_reason
    FROM checked WHERE len(quality_rule_ids) > 0
""")
show_query("SELECT * FROM staging_output ORDER BY unique_key LIMIT 2")
show_query("SELECT * FROM quarantine_output ORDER BY unique_key LIMIT 2")

unique_key,created_date,closed_date,complaint_type,agency,borough,status,open_data_channel_type,quality_missing_complaint_type,quality_missing_agency
63572326,2025-01-01 00:38:37,2025-01-01 01:14:24,Noise - Commercial,NYPD,MANHATTAN,Closed,ONLINE,False,False
63572331,2025-01-01 00:37:52,2025-01-01 00:50:50,Illegal Fireworks,NYPD,MANHATTAN,Closed,ONLINE,False,False


unique_key,created_date,closed_date,complaint_type,agency,borough,status,open_data_channel_type,quality_rule_ids,quarantine_reason
63656893,2025-01-06T07:51:00.000,2025-01-04T02:00:00.000,Street Light Condition,DOT,MANHATTAN,Pending,UNKNOWN,['valid_closure_order'],Дата закрытия не распознана или раньше создания
63748892,2025-01-13T09:22:00.000,2025-01-10T23:50:00.000,Street Light Condition,DOT,MANHATTAN,Pending,UNKNOWN,['valid_closure_order'],Дата закрытия не распознана или раньше создания


[('63656893',
  '2025-01-06T07:51:00.000',
  '2025-01-04T02:00:00.000',
  'Street Light Condition',
  'DOT',
  'MANHATTAN',
  'Pending',
  'UNKNOWN',
  ['valid_closure_order'],
  'Дата закрытия не распознана или раньше создания'),
 ('63748892',
  '2025-01-13T09:22:00.000',
  '2025-01-10T23:50:00.000',
  'Street Light Condition',
  'DOT',
  'MANHATTAN',
  'Pending',
  'UNKNOWN',
  ['valid_closure_order'],
  'Дата закрытия не распознана или раньше создания')]

### Запись и повторное чтение

Оба Parquet-файла записываются напрямую в существующий бакет staging.
При повторном запуске перезаписываются те же два ключа, новые партиции и дубликаты не добавляются.
Пустой карантин также записывается с определённой схемой

In [15]:
def sql_literal(value):
    return "'" + value.replace("'", "''") + "'"


con.execute(f"""
    COPY (SELECT * FROM staging_output ORDER BY unique_key)
    TO {sql_literal(STAGING_PATH)} (FORMAT PARQUET, COMPRESSION ZSTD)
""")
con.execute(f"""
    COPY (SELECT * FROM quarantine_output ORDER BY unique_key, created_date, closed_date)
    TO {sql_literal(QUARANTINE_PATH)} (FORMAT PARQUET, COMPRESSION ZSTD)
""")
con.read_parquet(STAGING_PATH, hive_partitioning=False).create_view("saved_staging")
con.read_parquet(QUARANTINE_PATH, hive_partitioning=False).create_view("saved_quarantine")
staging_count = con.execute("SELECT count(*) FROM saved_staging").fetchone()[0]
quarantine_count = con.execute("SELECT count(*) FROM saved_quarantine").fetchone()[0]
show_rows(
    ["Слой", "Строки", "Путь"],
    [
        ["raw", raw_count, RAW_PATH],
        ["staging", staging_count, STAGING_PATH],
        ["quarantine", quarantine_count, QUARANTINE_PATH],
    ],
)
assert raw_count == staging_count + quarantine_count

Слой,Строки,Путь
raw,61331,s3://raw/nyc311_manhattan/ingested_on=2026-10-14/nyc311_manhattan_2025-01.csv
staging,61304,s3://staging/nyc311_manhattan/source_ingested_on=2026-10-14/staging.parquet
quarantine,27,s3://staging/nyc311_manhattan/source_ingested_on=2026-10-14/quarantine.parquet


## Проверка сохранённых результатов

На заново прочитанном staging повторяются критические проверки.
Также сверяется каждый флаг с условием нарушения среди оставшихся записей.
Для карантина проверяются исходные значения с учётом повторов и наличие причин

In [16]:
critical_checks = con.execute(
    """
    SELECT
        (SELECT count(*) FROM saved_staging
            WHERE unique_key IS NULL OR NOT regexp_full_match(unique_key, '[0-9]+'))
        + (SELECT coalesce(sum(n - 1), 0) FROM (
            SELECT count(*) AS n FROM saved_staging GROUP BY unique_key HAVING count(*) > 1
          )) AS invalid_keys,
        count(*) FILTER (WHERE created_date IS NULL OR created_date < CAST(? AS TIMESTAMP)
            OR created_date >= CAST(? AS TIMESTAMP)) AS invalid_created,
        count(*) FILTER (
            WHERE closed_date IS NOT NULL AND closed_date < created_date
        ) AS invalid_closure
    FROM saved_staging
""",
    [PERIOD_START, PERIOD_END],
).fetchone()
show_rows(
    ["Некорректные ключи", "Некорректная дата создания", "Закрытие раньше создания"],
    [critical_checks],
)
assert all(value == 0 for value in critical_checks)
flag_checks = con.execute("""
    SELECT count(*) FILTER (WHERE complaint_type IS NULL) AS missing_type,
        count(*) FILTER (WHERE quality_missing_complaint_type) AS type_flags,
        count(*) FILTER (
            WHERE (complaint_type IS NULL) IS DISTINCT FROM quality_missing_complaint_type
        ) AS type_mismatches,
        count(*) FILTER (WHERE agency IS NULL) AS missing_agency,
        count(*) FILTER (WHERE quality_missing_agency) AS agency_flags,
        count(*) FILTER (
            WHERE (agency IS NULL) IS DISTINCT FROM quality_missing_agency
        ) AS agency_mismatches
    FROM saved_staging
""").fetchone()
show_rows(
    [
        "Пропуск темы",
        "Флаг темы",
        "Несовпадения темы",
        "Пропуск агентства",
        "Флаг агентства",
        "Несовпадения агентства",
    ],
    [flag_checks],
)
assert flag_checks[0] == flag_checks[1] and flag_checks[2] == 0
assert flag_checks[3] == flag_checks[4] and flag_checks[5] == 0
show_query("""
    SELECT unique_key, complaint_type, agency,
        quality_missing_complaint_type, quality_missing_agency
    FROM saved_staging WHERE quality_missing_complaint_type OR quality_missing_agency
    ORDER BY unique_key LIMIT 2
""")
show_query("SELECT * FROM saved_quarantine ORDER BY unique_key LIMIT 2")

Некорректные ключи,Некорректная дата создания,Закрытие раньше создания
0,0,0


Пропуск темы,Флаг темы,Несовпадения темы,Пропуск агентства,Флаг агентства,Несовпадения агентства
0,0,0,0,0,0


unique_key,complaint_type,agency,quality_missing_complaint_type,quality_missing_agency


unique_key,created_date,closed_date,complaint_type,agency,borough,status,open_data_channel_type,quality_rule_ids,quarantine_reason
63656893,2025-01-06T07:51:00.000,2025-01-04T02:00:00.000,Street Light Condition,DOT,MANHATTAN,Pending,UNKNOWN,['valid_closure_order'],Дата закрытия не распознана или раньше создания
63748892,2025-01-13T09:22:00.000,2025-01-10T23:50:00.000,Street Light Condition,DOT,MANHATTAN,Pending,UNKNOWN,['valid_closure_order'],Дата закрытия не распознана или раньше создания


[('63656893',
  '2025-01-06T07:51:00.000',
  '2025-01-04T02:00:00.000',
  'Street Light Condition',
  'DOT',
  'MANHATTAN',
  'Pending',
  'UNKNOWN',
  ['valid_closure_order'],
  'Дата закрытия не распознана или раньше создания'),
 ('63748892',
  '2025-01-13T09:22:00.000',
  '2025-01-10T23:50:00.000',
  'Street Light Condition',
  'DOT',
  'MANHATTAN',
  'Pending',
  'UNKNOWN',
  ['valid_closure_order'],
  'Дата закрытия не распознана или раньше создания')]

In [17]:
quarantine_columns = (
    "unique_key, created_date, closed_date, complaint_type, agency, borough, "
    "status, open_data_channel_type, quality_rule_ids, quarantine_reason"
)
quarantine_diff = con.execute(f"""
    SELECT count(*) FROM (
        (SELECT {quarantine_columns} FROM saved_quarantine
         EXCEPT ALL SELECT {quarantine_columns} FROM quarantine_output)
        UNION ALL
        (SELECT {quarantine_columns} FROM quarantine_output
         EXCEPT ALL SELECT {quarantine_columns} FROM saved_quarantine)
    )
""").fetchone()[0]
assert quarantine_diff == 0
assert (
    con.execute("""
    SELECT count(*) FROM saved_quarantine
    WHERE len(quality_rule_ids) = 0 OR nullif(trim(quarantine_reason), '') IS NULL
""").fetchone()[0]
    == 0
)
expected_quarantine_count = con.execute(
    "SELECT count(*) FROM checked WHERE len(quality_rule_ids) > 0"
).fetchone()[0]
assert expected_quarantine_count == quarantine_count
raw_after = raw_fingerprint()
assert raw_before == raw_after
run_summary = {
    "raw_rows": raw_count,
    "staging_rows": staging_count,
    "quarantine_rows": quarantine_count,
    "raw_sha256": raw_after["sha256"],
    "raw_unchanged": raw_before == raw_after,
    "critical_violations_in_staging": list(map(int, critical_checks)),
    "flag_mismatches": int(flag_checks[2] + flag_checks[5]),
    "quarantine_differences": quarantine_diff,
}
print(run_summary)
con.close()

{'raw_rows': 61331, 'staging_rows': 61304, 'quarantine_rows': 27, 'raw_sha256': '645bfe253cf4705f35d9381cc26d456b722c18e0bbc96562b4d7e54f247294d8', 'raw_unchanged': True, 'critical_violations_in_staging': [0, 0, 0], 'flag_mismatches': 0, 'quarantine_differences': 0}


## Решения и ограничения

Строки без даты закрытия сохраняются с NULL и не подходят для расчёта времени до закрытия.
Противоречивые даты не исправляются, запись сохраняется в карантине вместе с исходными значениями.
В staging нет нарушений критических правил, количество строк сохраняется в балансе с карантином

Исключение обращений в карантин уменьшает будущие подсчёты по staging относительно raw.
Набор отражает состояние динамического источника на дату выгрузки, дата закрытия не доказывает качество решения проблемы.
Восемь полей среза не позволяют анализировать community board из полного Project Brief

Краткий отчёт находится в `docs/data_quality_report.md`